# Phase 2 — Prithvi-100M Feature Extraction

Stream HLS granules directly from NASA S3, run them through Prithvi-EO-1.0-100M, and save 768-dim feature vectors to S3.

**Run this notebook on an EC2 GPU instance** (`g4dn.xlarge` recommended). CPU is feasible for small tests but too slow for all 5 states × 20 years.


In [ ]:
import sys
sys.path.insert(0, "../src")

from hls_downloader import login, iter_granules_cloud
from privthi_extractor import load_prithvi, extract_features_from_array, save_features_to_s3
import torch

BUCKET = "cornsight-data"

# Use 'cuda' if on EC2 GPU, 'cpu' for local testing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

login()
model = load_prithvi(device=DEVICE)


## Quick test — 2 granules for Iowa 2023

Run this before the full extraction to confirm the pipeline works end-to-end.


In [ ]:
for stacked, profile, gid in iter_granules_cloud("IA", 2023, "final", max_granules=2):
    features = extract_features_from_array(model, stacked, max_tiles=4)
    print(f"{gid}: features shape={features.shape}  mean={features.mean():.4f}")
    save_features_to_s3(features, gid, BUCKET, "IA", 2023, "final")
    print(f"  Saved to s3://{BUCKET}/processed/features/IA/2023/final/{gid}.npy")


## Full extraction — all 5 states × 2015–2024 × final forecast date

Run on EC2. Each state/year takes ~10–30 min on GPU depending on granule count.
Already-saved granules are skipped automatically.


In [ ]:
from privthi_extractor import list_features_in_s3

STATES = ["IA", "CO", "WI", "MO", "NE"]
YEARS = range(2015, 2025)
FORECAST_DATE = "final"

for state in STATES:
    for year in YEARS:
        print(f"\n=== {state} {year} [{FORECAST_DATE}] ===")
        # Check how many features already saved
        existing = list_features_in_s3(BUCKET, state, year, FORECAST_DATE)
        print(f"  Already saved: {len(existing)} granules")

        count = 0
        for stacked, profile, gid in iter_granules_cloud(state, year, FORECAST_DATE):
            s3_key = f"processed/features/{state}/{year}/{FORECAST_DATE}/{gid}.npy"
            if any(gid in k for k in existing):
                continue  # skip already processed
            try:
                features = extract_features_from_array(model, stacked)
                save_features_to_s3(features, gid, BUCKET, state, year, FORECAST_DATE)
                count += 1
            except Exception as e:
                print(f"  WARN {gid}: {e}")

        print(f"  Newly saved: {count} granules")
